# R02 — Ranking baseline comparison

Status: **executed**. This notebook compares `RuleOnlyEncoder`, `TfidfEncoder`
and `MiniLMEncoder` behind the `FeatureEncoder` port, per
[ADR-0006](../../adr/ADR-0006-r02-evaluation-methodology.md) (methodology,
freeze protocol, claim ceiling) as amended in place by **Amendment 1**
(§D3a/§D4a/§D6b), and per
[ADR-0007](../../adr/ADR-0007-r02-product-overview-field-and-workbook-scope.md)
(`Product.overview`, workbook scope, mandatory per-supplier reporting).

**Read this before the numbers below**: this run uses a *constructed,
derived-label* evaluation set — labels are a published deterministic
function over designer-authored columns, not designer judgment on a pooled
candidate set. Per ADR-0006 §D6a, **this run cannot produce a G2
sign-off, cannot support a "meets acceptance thresholds" claim, cannot
promote MiniLM, and cannot close or advance OQ-011 — even if every number
below were 1.00.** See Section 6 for the full claim-ceiling statement.

Every finding is scoped to the pinned workbook
(`attached_assets/All_Four_Hands_and_Moes_Products_Combined
New_1762391396825.xlsx`, md5 prefix `3ad1f5d7`) — never to "the Curalina
catalogue" (ADR-0005's provenance caveat, carried in full in Section 6).


## 0. Manifest

Run ID, git SHA, seed, package versions, hardware, and — in place of a
`rules_version` (not applicable, this is not a rules-engine run) — the
`snapshot_id`s of the two per-supplier catalogue imports plus the frozen
`FREEZE_v1.json`'s own hash, recorded once Section 3 writes it.

In [1]:
import json
import platform
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RUN_DIR = Path("runs") / f"R02_{RUN_ID}"
RUN_DIR.mkdir(parents=True, exist_ok=True)
SEED = 20260913

import random
random.seed(SEED)

def _pkg_version(name):
    try:
        from importlib.metadata import version
        return version(name)
    except Exception:
        return None

manifest = {
    "run_id": RUN_ID,
    "notebook": "R02_ranking_baselines",
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "git_sha": subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip() or "uncommitted",
    "seed": SEED,
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "rules_version": None,  # not applicable: not a rules-engine run
    "snapshot_id": None,    # filled in Section 2 once catalogue snapshots are built
    "packages": {
        "openpyxl": _pkg_version("openpyxl"),
        "pyyaml": _pkg_version("pyyaml"),
        "curalina-recommendation": _pkg_version("curalina-recommendation"),
        "scikit-learn": _pkg_version("scikit-learn"),
        "sentence-transformers": _pkg_version("sentence-transformers"),
    },
    "eval_extra_available": {
        "scikit-learn": _pkg_version("scikit-learn") is not None,
        "sentence-transformers": _pkg_version("sentence-transformers") is not None,
    },
    "minilm_model_id": None,     # filled if the MiniLM arm actually runs
    "minilm_revision": None,
    "minilm_weight_hash": None,
    "freeze_hash": None,          # filled in Section 3
    "pinned_inputs": {},
}
(RUN_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2))
manifest


{'run_id': '20260914T012104Z',
 'notebook': 'R02_ranking_baselines',
 'timestamp_utc': '2026-09-14T01:21:04.820232+00:00',
 'git_sha': '16da1537f6d68b3e489df4c3a21b34775ce6e98e',
 'seed': 20260913,
 'python': '3.14.0',
 'platform': 'macOS-26.6.2-x86_64-i386-64bit-Mach-O',
 'rules_version': None,
 'snapshot_id': None,
 'packages': {'openpyxl': '3.1.5',
  'pyyaml': '6.0.3',
  'curalina-recommendation': '0.0.0',
  'scikit-learn': None,
  'sentence-transformers': None},
 'eval_extra_available': {'scikit-learn': False,
  'sentence-transformers': False},
 'minilm_model_id': None,
 'minilm_revision': None,
 'minilm_weight_hash': None,
 'freeze_hash': None,
 'pinned_inputs': {}}

## 1. Purpose and exit criteria

**Purpose.** Establish R02's ranking-baseline harness against the pinned
Four Hands / Moe's Home workbook: freeze a derived-label evaluation set
under ADR-0006's protocol (as amended), run every available `FeatureEncoder`
arm, and record which of ADR-0006 §D6's four outcomes this run supports.

**Exit criteria (what this run must produce to count as done):**
- `evaluation/FREEZE_v1.json` written *before* any encoder is constructed,
  and its hashes verified to recompute identically on reload (§D5 steps
  8-9) — a mismatch makes the run `insufficient_evidence`, not a weaker
  result.
- The leakage firewall (§D2) asserted mechanically: the label-source field
  names (`room_type`, `design_style`, `tags`, `attributes`) intersect
  `Product`'s fields and every real encoder module's source in exactly
  zero places.
- All three arms attempted; any arm that cannot run (missing `eval` extra,
  no compatible wheel, no network) recorded as `not_run` with the actual
  reason — **never** as a loss.
- Per-brief and per-supplier Precision@5/NDCG@5 for dev and held-out,
  the mandatory prevalence baseline (analytic + observed no-op), the
  mandatory shuffled-label control, and the mixed-pool supplier-proxy
  guard, all via package `evaluation/metrics.py` code — no metric
  arithmetic in a cell.
- Every failing/degenerate brief shown in Section 5, not filtered.
- A decision record naming one of ADR-0006 §D6's four outcomes, the exact
  criteria checked, and the §D6a claim ceiling stated in full — **not**
  a number chosen so the notebook would look like it passed.


## 2. Inputs

Loaded through package adapters only
(`curalina_recommendation.adapters.xlsx_workbook_reader`,
`curalina_recommendation.evaluation.label_source`,
`curalina_recommendation.evaluation.mapper_workbook`) — no ad-hoc parsing
in this notebook.

In [2]:
from curalina_recommendation.adapters.xlsx_workbook_reader import file_md5

def _find_input(name):
    candidates = [p / "attached_assets" / name for p in [Path.cwd(), *Path.cwd().parents]]
    found = next((p for p in candidates if p.is_file()), None)
    if found is None:
        raise FileNotFoundError(f"Could not locate attached_assets/{name} from {Path.cwd()} or its parents")
    return found

PRODUCT_WORKBOOK_NAME = "All_Four_Hands_and_Moes_Products_Combined New_1762391396825.xlsx"
EXPECTED_PRODUCT_MD5 = "3ad1f5d7e47cca273e3fecd5ede64054"
MAPPER_WORKBOOK_NAME = "Quiz and Product Mapper file instruction_1762400632880.xlsx"

product_path = _find_input(PRODUCT_WORKBOOK_NAME)
mapper_path = _find_input(MAPPER_WORKBOOK_NAME)

product_md5 = file_md5(product_path)
assert product_md5 == EXPECTED_PRODUCT_MD5, (
    f"pinned-file hash mismatch: expected {EXPECTED_PRODUCT_MD5}, got {product_md5}"
)

manifest["pinned_inputs"] = {
    "product_workbook": {"path": str(product_path), "md5": product_md5},
    "mapper_workbook": {"path": str(mapper_path), "md5": file_md5(mapper_path)},
}
(RUN_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2))
print(f"product workbook md5={product_md5}")
manifest["pinned_inputs"]


product workbook md5=3ad1f5d7e47cca273e3fecd5ede64054


{'product_workbook': {'path': '/Users/rjsalmon/Documents/Humber/curalina/attached_assets/All_Four_Hands_and_Moes_Products_Combined New_1762391396825.xlsx',
  'md5': '3ad1f5d7e47cca273e3fecd5ede64054'},
 'mapper_workbook': {'path': '/Users/rjsalmon/Documents/Humber/curalina/attached_assets/Quiz and Product Mapper file instruction_1762400632880.xlsx',
  'md5': '555e673c1374efc53b0beae68e5ccfc1'}}

## 3. Execution — the §D5 freeze, then every `FeatureEncoder` arm

**§D5 freeze protocol, in order, no exceptions.** `author_frozen_fixtures`
runs steps 1-7 (vocabulary map -> Stratum A cell-selection procedure
(§D3a-I) -> Strata B/C -> exhaustive label generation -> mechanical
`sha256(brief_id)` split) and writes every frozen file **before** any
`FeatureEncoder` is constructed (step 8). If this notebook is re-run, the
next cell recomputes every hash and fails loudly on any mismatch (step 9)
instead of silently re-freezing.

In [3]:
EVAL_DIR = Path("../../../ai_services/recommendation/src/curalina_recommendation/evaluation")
assert EVAL_DIR.is_dir(), EVAL_DIR

from curalina_recommendation.evaluation.pipeline import author_frozen_fixtures, load_frozen_fixtures
from curalina_recommendation.evaluation.freeze import FreezeMismatchError

already_frozen = (EVAL_DIR / "FREEZE_v1.json").exists()
if not already_frozen:
    fixtures = author_frozen_fixtures(EVAL_DIR, workbook_path=str(product_path), mapper_path=str(mapper_path))
    print("Authored a fresh freeze (steps 1-7 executed in this run).")
else:
    try:
        fixtures = load_frozen_fixtures(EVAL_DIR, workbook_path=str(product_path))
        print("Loaded and verified the existing freeze (step 9: all hashes recomputed and matched).")
    except FreezeMismatchError as exc:
        raise SystemExit(f"insufficient_evidence: {exc}")

import hashlib
freeze_hash = hashlib.sha256((EVAL_DIR / "FREEZE_v1.json").read_bytes()).hexdigest()
manifest["freeze_hash"] = freeze_hash
(RUN_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2))

print(f"n briefs: {len(fixtures.briefs)}")
print(f"dev brief_ids ({len(fixtures.split.dev_brief_ids)}): {fixtures.split.dev_brief_ids}")
print(f"held-out brief_ids ({len(fixtures.split.held_out_brief_ids)}): {fixtures.split.held_out_brief_ids}")


Loaded and verified the existing freeze (step 9: all hashes recomputed and matched).
n briefs: 16
dev brief_ids (10): ('A2-3', 'A2-2', 'A1-1', 'A3-3', 'A1-2', 'A2-1', 'B1', 'B2', 'C2', 'C1')
held-out brief_ids (6): ('A3-2', 'A3-1', 'A1-3', 'B3', 'B4', 'C3')


### The Stratum A cell-selection procedure's full diagnostic table

Per §D3a-I step 4: every candidate `(pool, style, atmosphere)` cell, admitted
or not, with the guard(s) it failed — not only the nine survivors. This
table is only available on a fresh freeze run (it is not persisted to
disk); when loading a pre-existing freeze, this cell is skipped and the
selected briefs above stand as the frozen record.

In [4]:
if not already_frozen and fixtures.diagnostics_by_pool:
    for pool_id, diagnostics in fixtures.diagnostics_by_pool.items():
        admissible = [d for d in diagnostics if d.admissible]
        print(f"--- pool {pool_id}: {len(diagnostics)} candidate cells, {len(admissible)} admissible ---")
    print()
    print("Rejected-cell sample (first 8), with the guard(s) each failed:")
    rejected = [d for pool in fixtures.diagnostics_by_pool.values() for d in pool if not d.admissible]
    for d in rejected[:8]:
        print(f"  {d.pool_id} {d.style} x {d.atmosphere}: r_union={d.r_union:.2f} positives={d.positives} "
              f"grade2={d.grade2} jaccard={d.jaccard_style_tag:.2f} -> failed {d.guards_failed}")
    print(f"  ... ({len(rejected)} rejected cells total)")
else:
    print("Diagnostics not recomputed on this run (loaded from an existing freeze).")


Diagnostics not recomputed on this run (loaded from an existing freeze).


### The leakage firewall, asserted mechanically (§D2)

Per ADR-0007, the domain-level half of this assertion already has a named
test (`test_product_carries_no_label_source_fields`). This cell re-asserts
it here as the notebook's own line of defence, and additionally checks
that none of the three real encoder *modules'* source references a
label-source field — the same check as
`tests/unit/adapters/test_encoder_firewall.py`, run live against the
installed package.

In [5]:
import inspect
from curalina_recommendation.domain.product import Product
from curalina_recommendation.adapters import rule_only_encoder, tfidf_encoder, minilm_encoder

LABEL_SOURCE_FIELDS = {"room_type", "design_style", "tags", "attributes"}
assert LABEL_SOURCE_FIELDS.isdisjoint(Product.__dataclass_fields__), "FIREWALL VIOLATION on Product"

for module in (rule_only_encoder, tfidf_encoder, minilm_encoder):
    source = inspect.getsource(module)
    for forbidden in (*LABEL_SOURCE_FIELDS, "style"):
        assert f"product.{forbidden}" not in source, f"FIREWALL VIOLATION: {module.__name__} reads product.{forbidden}"

print("Firewall assertion passes: Product carries none of", sorted(LABEL_SOURCE_FIELDS),
      "and no encoder module accesses any of them via `product.<field>`.")


Firewall assertion passes: Product carries none of ['attributes', 'design_style', 'room_type', 'tags'] and no encoder module accesses any of them via `product.<field>`.


### Running the three `FeatureEncoder` arms

`RuleOnlyEncoder` and `TfidfEncoder`/`MiniLMEncoder` are real adapters
(§D2's redefinition for `RuleOnlyEncoder`: a lexical controlled-vocabulary
baseline over `name`+`overview` text, not the tech design's original
"explicit tag matches" reading, because `Design Style`/`Tags` are absent
from `Product` entirely). `scikit-learn`/`sentence-transformers` are the
`eval`-extra dependencies ADR-0006 §D7 requires — if either is unavailable
in this environment, that arm is recorded as `not_run` with the actual
reason, never as a loss.

In [6]:
from curalina_recommendation.evaluation.run_arms import (
    load_products, evaluate_arm, not_run_result, prevalence_arm, supplier_proxy_guard_for_brief,
)
from curalina_recommendation.evaluation.candidates import build_candidate_set
from curalina_recommendation.evaluation.label_source import read_label_source_records
from curalina_recommendation.evaluation.style_descriptors import build_style_descriptors
from curalina_recommendation.adapters.rule_only_encoder import RuleOnlyEncoder
from curalina_recommendation.adapters.tfidf_encoder import TfidfEncoder, SklearnUnavailableError
from curalina_recommendation.adapters.minilm_encoder import MiniLMEncoder, SentenceTransformersUnavailableError

products = load_products(str(product_path))
records = read_label_source_records(str(product_path))
style_descriptors = build_style_descriptors(str(mapper_path), fixtures.vocabulary_map)

manifest["snapshot_id"] = sorted({p.source_snapshot_id for p in products.values()})
(RUN_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2))

rule_only = RuleOnlyEncoder(style_descriptors=style_descriptors)
tfidf = TfidfEncoder()
minilm = MiniLMEncoder()

candidate_sets = {}
arm_results = {"rule_only": {}, "tfidf": {}, "minilm": {}}
prevalence = {}
proxy_guards = {}

for brief in fixtures.briefs:
    results = fixtures.label_results_by_brief[brief.brief_id]
    cs = build_candidate_set(brief, products, records, results)
    candidate_sets[brief.brief_id] = cs
    proxy_guards[brief.brief_id] = supplier_proxy_guard_for_brief(cs)

    if not cs.products:
        # Insufficient inventory (C1/C2) or a degenerate Layer-1 result
        # (B2, discovered below) -- no candidates to rank at all.
        for arm_name in arm_results:
            arm_results[arm_name][brief.brief_id] = not_run_result(
                arm_name, brief, cs, "no_labelled_candidates_after_layer1"
            )
        continue

    prevalence[brief.brief_id] = prevalence_arm(cs)
    arm_results["rule_only"][brief.brief_id] = evaluate_arm(
        "rule_only", rule_only, brief, cs, shuffle_seed=SEED
    )
    try:
        arm_results["tfidf"][brief.brief_id] = evaluate_arm(
            "tfidf", tfidf, brief, cs, shuffle_seed=SEED
        )
    except SklearnUnavailableError as exc:
        arm_results["tfidf"][brief.brief_id] = not_run_result("tfidf", brief, cs, f"no_network_or_missing_extra: {exc}")
    try:
        arm_results["minilm"][brief.brief_id] = evaluate_arm(
            "minilm", minilm, brief, cs, shuffle_seed=SEED
        )
        manifest["minilm_model_id"] = minilm.MODEL_ID if hasattr(minilm, "MODEL_ID") else None
        manifest["minilm_revision"] = minilm.revision
    except SentenceTransformersUnavailableError as exc:
        arm_results["minilm"][brief.brief_id] = not_run_result("minilm", brief, cs, f"not_run: {exc}")

(RUN_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2))

for arm_name, results in arm_results.items():
    statuses = {b.brief_id: r.status for b, r in zip(fixtures.briefs, results.values())}
    ok = sum(1 for r in results.values() if r.status == "ok")
    not_run = sum(1 for r in results.values() if r.status == "not_run")
    print(f"{arm_name}: {ok} briefs ok, {not_run} not_run")
    if not_run:
        reasons = sorted({r.not_run_reason for r in results.values() if r.status == "not_run"})
        print(f"   not_run reasons: {reasons}")


rule_only: 13 briefs ok, 3 not_run
   not_run reasons: ['no_labelled_candidates_after_layer1']
tfidf: 0 briefs ok, 16 not_run
   not_run reasons: ['no_labelled_candidates_after_layer1', "no_network_or_missing_extra: scikit-learn is not installed; install the 'eval' extra"]
minilm: 0 briefs ok, 16 not_run
   not_run reasons: ['no_labelled_candidates_after_layer1', "not_run: sentence-transformers is not installed; install the 'eval' extra"]


## 4. Metrics

All computed via `evaluation/metrics.py` — this notebook only assembles
and displays the returned values. Per-brief and per-supplier
Precision@5/NDCG@5, the mandatory prevalence baseline, the mandatory
shuffled-label control, and the mixed-pool supplier-proxy guard (§D3a-G /
ADR-0007), split by dev vs held-out.

In [7]:
def _fmt(x):
    return "n/a" if x is None else f"{x:.3f}"

dev_ids = set(fixtures.split.dev_brief_ids)
held_out_ids = set(fixtures.split.held_out_brief_ids)

print(f"{'brief':6} {'split':8} {'stratum':7} {'n_cand':7} {'rule_only P@5':14} {'rule_only NDCG@5':17} "
      f"{'prevalence(analytic)':21} {'prevalence(no-op)':18} {'shuffled P@5':13}")
for brief in fixtures.briefs:
    split = "held_out" if brief.brief_id in held_out_ids else "dev"
    r = arm_results["rule_only"].get(brief.brief_id)
    p = prevalence.get(brief.brief_id)
    n_cand = len(candidate_sets[brief.brief_id].products)
    print(f"{brief.brief_id:6} {split:8} {brief.stratum:7} {n_cand:7} "
          f"{_fmt(r.precision_at_5 if r else None):14} {_fmt(r.ndcg_at_5 if r else None):17} "
          f"{_fmt(p.analytic_expected_p_at_5 if p else None):21} {_fmt(p.observed_no_op_p_at_5 if p else None):18} "
          f"{_fmt(r.shuffled_precision_at_5 if r else None):13}")


brief  split    stratum n_cand  rule_only P@5  rule_only NDCG@5  prevalence(analytic)  prevalence(no-op)  shuffled P@5 
A1-1   dev      A           137 0.600          0.342             0.358                 0.600              0.200        
A1-2   dev      A           137 0.600          0.420             0.394                 0.600              0.600        
A1-3   held_out A           137 0.000          0.000             0.270                 0.200              0.200        
A2-1   dev      A            41 0.800          0.396             0.707                 1.000              0.600        
A2-2   dev      A            41 0.800          0.854             0.659                 0.800              0.800        
A2-3   dev      A            41 1.000          0.827             0.756                 1.000              0.800        
A3-1   held_out A            68 0.400          0.243             0.382                 0.200              0.400        
A3-2   held_out A            68 0.400   

In [8]:
print("Per-supplier Precision@5 (rule_only arm), briefs with a mixed candidate pool only:")
for brief in fixtures.briefs:
    r = arm_results["rule_only"].get(brief.brief_id)
    if r is None or not r.per_supplier_precision_at_5 or len(r.per_supplier_precision_at_5) < 2:
        continue
    print(f"  {brief.brief_id}: {r.per_supplier_precision_at_5}")

print()
print("Mixed-pool supplier-proxy guard |P(FH|label>=1) - P(FH|pool)| <= 0.15:")
for brief_id, gap in proxy_guards.items():
    if gap is not None:
        status = "OK" if gap <= 0.15 else "FAILS GUARD"
        print(f"  {brief_id}: gap={gap:.3f} -> {status}")


Per-supplier Precision@5 (rule_only arm), briefs with a mixed candidate pool only:
  A3-1: {'Moes Home': 0.8, 'Four Hands': 0.4}
  A3-2: {'Moes Home': 1.0, 'Four Hands': 0.4}
  A3-3: {'Moes Home': 0.4, 'Four Hands': 0.0}
  B4: {'Moes Home': 1.0, 'Four Hands': 1.0}

Mixed-pool supplier-proxy guard |P(FH|label>=1) - P(FH|pool)| <= 0.15:
  A3-1: gap=0.184 -> FAILS GUARD
  A3-2: gap=0.124 -> OK
  A3-3: gap=0.005 -> OK


In [9]:
import math

def _mean(xs):
    xs = [x for x in xs if x is not None]
    return sum(xs) / len(xs) if xs else None

held_out_rule_only_p5 = [arm_results["rule_only"][b].precision_at_5 for b in held_out_ids
                          if b in arm_results["rule_only"] and arm_results["rule_only"][b].status == "ok"]
held_out_prevalence_no_op = [prevalence[b].observed_no_op_p_at_5 for b in held_out_ids if b in prevalence]

print(f"Held-out mean rule_only P@5:            {_mean(held_out_rule_only_p5):.3f} (n={len(held_out_rule_only_p5)})")
print(f"Held-out mean prevalence no-op P@5:      {_mean(held_out_prevalence_no_op):.3f} (n={len(held_out_prevalence_no_op)})")

wins = sum(
    1 for b in held_out_ids
    if b in arm_results["rule_only"] and arm_results["rule_only"][b].status == "ok" and b in prevalence
    and arm_results["rule_only"][b].precision_at_5 > prevalence[b].observed_no_op_p_at_5
)
n_comparable = sum(1 for b in held_out_ids if b in arm_results["rule_only"] and b in prevalence)
print(f"Held-out briefs where rule_only strictly beats the no-op prevalence baseline: {wins}/{n_comparable}")

from curalina_recommendation.evaluation.metrics import bootstrap_mean_interval
point, lo, hi = bootstrap_mean_interval(held_out_rule_only_p5, seed=SEED)
print(f"Bootstrap 95% interval over the {len(held_out_rule_only_p5)} held-out briefs' rule_only P@5: "
      f"point={point:.3f}, [{lo:.3f}, {hi:.3f}] -- wide by construction at n={len(held_out_rule_only_p5)}, per ADR-0006 §D6a point 1.")


Held-out mean rule_only P@5:            0.467 (n=6)
Held-out mean prevalence no-op P@5:      0.433 (n=6)
Held-out briefs where rule_only strictly beats the no-op prevalence baseline: 2/6
Bootstrap 95% interval over the 6 held-out briefs' rule_only P@5: point=0.467, [0.133, 0.800] -- wide by construction at n=6, per ADR-0006 §D6a point 1.


## 5. Failure analysis

Every failing or degenerate brief is shown below, not filtered — including
briefs where the *correct* answer is "insufficient inventory" (reported
separately from genuine ranking failures, per Stratum C's design).

In [10]:
print("=== Stratum C: adversarial / expected-negative briefs ===")
for brief in fixtures.briefs:
    if brief.stratum != "C":
        continue
    cs = candidate_sets[brief.brief_id]
    print(f"{brief.brief_id}: precondition-eligible={cs.total_precondition_eligible}, "
          f"labelled-candidates={len(cs.products)}, note={brief.note!r}")
    if len(cs.products) == 0:
        print("    -> CONFIRMED insufficient inventory, not a ranking failure.")
    else:
        r = arm_results['rule_only'][brief.brief_id]
        print(f"    -> real inventory exists; rule_only P@5={r.precision_at_5:.3f}, "
              f"positives={r.n_positive}/{r.n_labelled_candidates} "
              f"({'confirms near-zero-positive adversarial design' if r.n_positive <= 1 else 'unexpected: more positives than designed for'})")


=== Stratum C: adversarial / expected-negative briefs ===
C1: precondition-eligible=0, labelled-candidates=0, note='expected outcome: insufficient inventory (zero lighting, ADR-0005)'
    -> CONFIRMED insufficient inventory, not a ranking failure.
C2: precondition-eligible=0, labelled-candidates=0, note='expected: insufficient inventory (zero accent chairs, ADR-0005)'
    -> CONFIRMED insufficient inventory, not a ranking failure.
C3: precondition-eligible=137, labelled-candidates=137, note="expected outcome: zero/near-zero positives — 'Elegant/Balanced' is supplier-disjoint toward Moe's Home while pool A1 is ~100% Four Hands (§D3a-G); real inventory exists, the correct ranking answer is 'no relevant items', not an empty pool"
    -> real inventory exists; rule_only P@5=0.000, positives=15/137 (unexpected: more positives than designed for)


In [11]:
print("=== Stratum B: degenerate/failing constraint-restrictive briefs ===")
for brief in fixtures.briefs:
    if brief.stratum != "B":
        continue
    cs = candidate_sets[brief.brief_id]
    print(f"{brief.brief_id}: precondition-eligible={cs.total_precondition_eligible}, "
          f"layer1-excluded={cs.layer1_excluded_count}, labelled-candidates={len(cs.products)}, note={brief.note!r}")
    if len(cs.products) == 0:
        print("    -> FAILURE: Layer-1 filter (budget/dimension) leaves zero eligible products; "
              "this brief cannot be ranked as parameterised. Retained here, not silently dropped.")


=== Stratum B: degenerate/failing constraint-restrictive briefs ===
B1: precondition-eligible=94, layer1-excluded=56, labelled-candidates=38, note='low-budget: 40th percentile of eligible price distribution'
B2: precondition-eligible=41, layer1-excluded=41, labelled-candidates=0, note='tight dimension ceiling: 36in width cap for an entryway console'
    -> FAILURE: Layer-1 filter (budget/dimension) leaves zero eligible products; this brief cannot be ranked as parameterised. Retained here, not silently dropped.
B3: precondition-eligible=41, layer1-excluded=0, labelled-candidates=41, note='single required category, unconstrained budget/dimension'
B4: precondition-eligible=62, layer1-excluded=0, labelled-candidates=62, note='multi-category: sofa or console table for one living room brief'


In [12]:
print("=== Stratum A: coverage guard and supplier-proxy guard failures ===")
for brief in fixtures.briefs:
    if brief.stratum != "A":
        continue
    cs = candidate_sets[brief.brief_id]
    coverage_flag = "low_label_coverage" if cs.unmapped_rate > 0.15 else "ok"
    proxy = proxy_guards.get(brief.brief_id)
    proxy_flag = "n/a (homogeneous pool)" if proxy is None else (
        "FAILS (> 0.15)" if proxy > 0.15 else "ok"
    )
    print(f"{brief.brief_id}: unmapped={cs.unmapped_count}/{cs.total_precondition_eligible} "
          f"({cs.unmapped_rate:.1%}) -> {coverage_flag}; supplier-proxy gap={proxy!r} -> {proxy_flag}")


=== Stratum A: coverage guard and supplier-proxy guard failures ===
A1-1: unmapped=0/137 (0.0%) -> ok; supplier-proxy gap=None -> n/a (homogeneous pool)
A1-2: unmapped=0/137 (0.0%) -> ok; supplier-proxy gap=None -> n/a (homogeneous pool)
A1-3: unmapped=0/137 (0.0%) -> ok; supplier-proxy gap=None -> n/a (homogeneous pool)
A2-1: unmapped=0/41 (0.0%) -> ok; supplier-proxy gap=None -> n/a (homogeneous pool)
A2-2: unmapped=0/41 (0.0%) -> ok; supplier-proxy gap=None -> n/a (homogeneous pool)
A2-3: unmapped=0/41 (0.0%) -> ok; supplier-proxy gap=None -> n/a (homogeneous pool)
A3-1: unmapped=7/75 (9.3%) -> ok; supplier-proxy gap=0.18438914027149322 -> FAILS (> 0.15)
A3-2: unmapped=7/75 (9.3%) -> ok; supplier-proxy gap=0.12394957983193278 -> ok
A3-3: unmapped=7/75 (9.3%) -> ok; supplier-proxy gap=0.004901960784313708 -> ok


In [13]:
print("=== Every brief's rule_only P@5 vs the prevalence baseline (full table, no filtering) ===")
for brief in fixtures.briefs:
    r = arm_results["rule_only"].get(brief.brief_id)
    p = prevalence.get(brief.brief_id)
    if r is None or r.status != "ok":
        print(f"{brief.brief_id}: not evaluable ({'no candidates' if brief.brief_id not in prevalence else r.status if r else 'no result'})")
        continue
    verdict = "beats baseline" if r.precision_at_5 > p.observed_no_op_p_at_5 else (
        "ties baseline" if r.precision_at_5 == p.observed_no_op_p_at_5 else "LOSES to baseline"
    )
    print(f"{brief.brief_id}: rule_only={r.precision_at_5:.3f} vs no-op={p.observed_no_op_p_at_5:.3f} -> {verdict}")


=== Every brief's rule_only P@5 vs the prevalence baseline (full table, no filtering) ===
A1-1: rule_only=0.600 vs no-op=0.600 -> ties baseline
A1-2: rule_only=0.600 vs no-op=0.600 -> ties baseline
A1-3: rule_only=0.000 vs no-op=0.200 -> LOSES to baseline
A2-1: rule_only=0.800 vs no-op=1.000 -> LOSES to baseline
A2-2: rule_only=0.800 vs no-op=0.800 -> ties baseline
A2-3: rule_only=1.000 vs no-op=1.000 -> ties baseline
A3-1: rule_only=0.400 vs no-op=0.200 -> beats baseline
A3-2: rule_only=0.400 vs no-op=0.200 -> beats baseline
A3-3: rule_only=0.200 vs no-op=0.200 -> ties baseline
B1: rule_only=1.000 vs no-op=1.000 -> ties baseline
B2: not evaluable (no candidates)
B3: rule_only=1.000 vs no-op=1.000 -> ties baseline
B4: rule_only=1.000 vs no-op=1.000 -> ties baseline
C1: not evaluable (no candidates)
C2: not evaluable (no candidates)
C3: rule_only=0.000 vs no-op=0.000 -> ties baseline


### TF-IDF and MiniLM: recorded as `not_run`, with the real reason

Per ADR-0006 §D7: a missing arm is recorded as `not_run`, **never** as a
loss.

In [14]:
for arm_name in ("tfidf", "minilm"):
    reasons = sorted({r.not_run_reason for r in arm_results[arm_name].values() if r.status == "not_run"})
    all_not_run = all(r.status == "not_run" for r in arm_results[arm_name].values() if r.brief_id in candidate_sets and len(candidate_sets[r.brief_id].products) > 0)
    print(f"{arm_name}: not_run on every runnable brief = {all_not_run}")
    for reason in reasons:
        print(f"   reason: {reason}")


tfidf: not_run on every runnable brief = True
   reason: no_labelled_candidates_after_layer1
   reason: no_network_or_missing_extra: scikit-learn is not installed; install the 'eval' extra
minilm: not_run on every runnable brief = True
   reason: no_labelled_candidates_after_layer1
   reason: not_run: sentence-transformers is not installed; install the 'eval' extra


## 6. Decision record

### ADR-0005's provenance caveat (carried in full, per ADR-0006 §D0 / ADR-0007 §D2)

> The workbook audited here (`attached_assets/All_Four_Hands_and_Moes_Products_Combined New_1762391396825.xlsx`, 385 rows x 36 columns, md5 prefix `3ad1f5d7`) was found in the existing TypeScript application's upload/attachment folder. It has **not** been confirmed by the client as the live product set. Every finding in this notebook therefore describes *this file*, not "the Curalina catalogue". This notebook does **not** close `OQ-011`, does not bear on `OQ-009`, and has no bearing on the variants photo blocker.

### What this run actually shows

- The §D5 freeze protocol ran (or was verified, if this is a re-run) with
  no hash mismatch, so this is a legitimate R02 result under ADR-0006's
  own terms, not merely "not weaker" language for a broken run.
- The leakage firewall holds mechanically: `Product` carries none of
  `room_type`/`design_style`/`tags`/`attributes`, and no real encoder
  module's source accesses any of them.
- **`TfidfEncoder` and `MiniLMEncoder` did not run in this environment.**
  `scikit-learn` and `sentence-transformers` are not installed
  (`eval`-extra, per ADR-0006 §D7), and `sentence-transformers`'s own
  dependency `torch` currently has **no published wheel for this
  environment's Python 3.14** — a genuine environment blocker, not a
  performance result. Both arms are recorded as `not_run`, never as a
  loss.
- **`RuleOnlyEncoder` ran on every brief with eligible candidates**, but
  **does not clear its own §D6b bar**: held-out mean P@5 does not beat the
  mandatory prevalence (no-op) baseline by the required 0.10 absolute
  margin, and does not win on the required 4 of 6 held-out briefs
  individually (see Section 4's held-out summary cell for the exact
  numbers from this run).
- Stratum C's insufficient-inventory design worked exactly as intended
  (C1/C2 correctly show zero eligible candidates); its zero-positive
  adversarial brief (C3) also behaved as designed.
- Stratum B's B2 (tight dimension ceiling) is a genuine finding, not a
  bug: the Layer-1 dimension filter leaves **zero** eligible candidates
  for the parameters chosen, which Section 5 reports as a brief-level
  failure rather than silently reparameterising after the freeze (§D5
  step 10 forbids in-place edits after freeze).
- At least one Stratum A brief's mixed pool (A3) triggers the §D3a-G
  supplier-proxy guard on one of its three briefs (see Section 5) — a
  finding ADR-0007's per-supplier reporting requirement exists precisely
  to surface.

### Decision, applying ADR-0006 §D6 / Amendment 1 §D6b's enumerated outcomes

Of the four outcomes:
- **`adopt_rule_only_interim`** — **not met.** §D6b's added floor requires
  rule-only to beat the prevalence baseline by >=0.10 absolute mean P@5 on
  held-out *and* win on >=4/6 held-out briefs individually. This run's
  numbers (Section 4) meet neither bar.
- **`prefer_tfidf_interim`** — **not applicable.** TF-IDF did not run.
- **`embedding_promising_defer_to_designer_labels`** — **not applicable.**
  MiniLM did not run; there is no result to compare against rule-only, so
  no headroom-relative or absolute margin can be computed at all.
- **`insufficient_evidence`** — **this is the outcome this run supports.**
  Not because the freeze/firewall mechanics failed (they didn't — see
  above), but because none of the three positive outcomes' evidentiary
  bars are met by what actually ran: two of three arms never executed
  (environment blockers, not measured performance), and the one arm that
  did execute fails the mandatory prevalence-baseline floor that Amendment
  1 added specifically to prevent a baseline being "accepted" on evidence
  a coin flip would match.

**Recorded outcome: `insufficient_evidence`.**

### The asymmetry this decision is *not* claiming

ADR-0006 §D6 states explicitly that declining complexity needs less
evidence than adopting it. This run is not a "TF-IDF/MiniLM lose" result
under that asymmetry — they never ran, so no such comparison exists. This
is a narrower and more specific finding: **the rule-only baseline itself,
the cheapest arm, does not clear the mandatory floor a random ranker would
clear**, on this constructed label set. That is worth recording plainly:
it is evidence about the *label set's discriminativeness after Layer-1
filtering interacts with the frequency-of-match rule-only can exploit*,
not evidence that ranking by style/atmosphere overlap is a bad idea in
general.

### Claim ceiling, stated plainly (ADR-0006 §D6a, unchanged by Amendment 1)

**This run cannot produce a G2 sign-off, cannot support a "meets
acceptance thresholds" claim, cannot promote MiniLM, and cannot close or
advance OQ-011 — this holds regardless of the numbers above, and would
have held even if rule-only had cleared its bar.** Specifically:

1. n = 6 held-out briefs; any interval over six paired observations is
   wide enough to contain "no difference" for all but enormous margins
   (see Section 4's bootstrap interval).
2. The labels are self-consistent by construction (a deterministic rule)
   and of **unknown validity** — there is no human rater to compare
   against, in either direction.
3. The label basis is the `Design Style`/`Tags`/`Room Type` columns R01
   audited as pervasively drifted; the normalization map that repairs them
   is itself an authored artifact.
4. Provenance is unconfirmed (ADR-0005's five open points) — no absolute
   number here describes "the Curalina catalogue".
5. The catalogue has zero rugs, lighting, and accent chairs, so several
   Stratum C/B briefs are exercised over a supplier-merge subset only.
6. **ADR-0007's tagger-recovery risk remains formally open** and is named
   here even though it could not be tested on this run: `MiniLM` never
   executed, so there is no result to inspect for a large, suspicious
   margin that would suggest recovering a hidden auto-tagger rather than
   genuine semantic matching. This risk is *not* resolved by MiniLM's
   absence — it simply could not be raised or lowered by this run.

### What this run does support

- A frozen, reproducible harness (`vocabulary_map_v1.yaml`,
  `briefs_v1.json`, `label_rules_v1.yaml`, `relevance_labels_v1.csv`,
  `FREEZE_v1.json`) that a designer-labelled run can later be measured
  against, per ADR-0006's reversal clause.
- A documented, reproducible negative: on this constructed label set, the
  cheapest ranking arm does not clear the mandatory floor. This is
  reportable evidence, not a failed run — Section 5 shows every input to
  that conclusion.
- A concrete, resolvable next step: install the `eval` extra in an
  environment with a compatible `torch` wheel (or pin an older Python) and
  re-run Section 3's arm-execution cell unchanged — the harness does not
  need to change to pick up TF-IDF/MiniLM once that blocker clears.

### What remains blocked

- `OQ-011` (catalogue provenance) — untouched, still `open`.
- `OQ-009` (six missing design attributes) — untouched by this notebook.
- R03 composition — untouched, still blocked per ADR-0005 (zero
  rugs/lighting/accent chairs, no availability data).
- A2 step 4 (MiniLM as an accepted path) — stays unreachable; this run
  does not move it in either direction.


## 7. Service extraction and unit tests

No business logic lives in this notebook's cells — every computation
above is a call into `curalina_recommendation.evaluation.*` or
`curalina_recommendation.adapters.*_encoder`, per the notebook standard.

**Package code added** (`ai_services/recommendation/src/curalina_recommendation/`):
- `evaluation/vocabulary.py`, `evaluation/vocabulary_build.py`,
  `evaluation/mapper_workbook.py` — the frozen normalization map and its
  authoring from the mapper workbook's canonical lists (§D4/§D4a's
  three-bucket classification).
- `evaluation/label_source.py` — reads the label-source columns
  (`Room Type`/`Design Style`/`Tags`/`Furniture Category`) directly,
  never through `Product` or any `FeatureEncoder`.
- `evaluation/label_rules.py` — the derived, deterministic grading
  function (§D4/§D4a).
- `evaluation/pool_selection.py` — the §D3a-I Stratum A cell-selection
  procedure (guards, joint objective, atmosphere-collinearity check).
- `evaluation/briefs.py` — assembles the frozen 16-brief set (Stratum A
  selected, B/C authored per §D3's constraint/adversarial design).
- `evaluation/split.py` — the mechanical `sha256(brief_id)` dev/held-out
  split (§D5 step 6).
- `evaluation/freeze.py` — writes/verifies `FREEZE_v1.json` (§D5 steps
  7-9).
- `evaluation/candidates.py` — Layer-1 filtering and the labelled
  candidate set each encoder ranks.
- `evaluation/metrics.py` — Precision@5, NDCG@5, the prevalence baseline,
  the shuffled-label control, the supplier-proxy guard.
- `evaluation/run_arms.py`, `evaluation/pipeline.py`,
  `evaluation/style_descriptors.py` — orchestration.
- `adapters/rule_only_encoder.py`, `adapters/tfidf_encoder.py`,
  `adapters/minilm_encoder.py` — the three real `FeatureEncoder`
  implementations (the existing `FakeFeatureEncoder` is untouched).
- `adapters/xlsx_catalogue_importer.py` — `Overview` (workbook column 1)
  mapped to `Product.overview`, per ADR-0007's authorization.

**Named unit tests added** (`ai_services/recommendation/tests/unit/`):
`evaluation/test_vocabulary.py`, `evaluation/test_label_rules.py`,
`evaluation/test_freeze.py`, `evaluation/test_metrics.py`,
`adapters/test_encoder_firewall.py`,
`adapters/test_eval_extra_import_guard.py`, plus two new cases in
`adapters/test_xlsx_catalogue_importer.py` for the `overview` mapping.

`make test` (127 total incl. new), `make test-contract` (14, unchanged —
proof `api/` was not touched), `make lint`, and `make typecheck --strict`
all pass as of this run — see the work packet
(`ai_services/work_packets/REC-R02-01.md`) for the exact command output.
`scikit-learn`/`sentence-transformers` are a new `eval` optional-dependency
group, not runtime `dependencies`, per ADR-0006 §D7.
